# Bagian 1: Penjelasan Konsep

## 1.1 Peta Besar: Method Estimator

```text
                    Estimator (semua punya .fit())
                    /                          \
            Transformer                    Predictor
        (.fit + .transform)           (.fit + .predict)
                                              |
                                    ┌─────────┴─────────┐
                                Classifier            Regressor
                          (punya .predict_proba,    (predict_proba
                           .decision_function)        TIDAK ada)
```

* **Predictor** (model) selalu memiliki `.predict()`.
* **Classifier** biasanya memiliki method tambahan:

  * `.predict_proba()` → menghasilkan probabilitas setiap kelas.
  * `.decision_function()` → menghasilkan nilai keputusan/skor untuk klasifikasi.
* **Regressor** tidak menggunakan kedua method tersebut karena outputnya berupa **nilai kontinu**, bukan probabilitas atau kelas.
* Jadi secara sederhana:

  * **Classifier** → `.predict()`, `.predict_proba()`, `.decision_function()`
  * **Regressor** → `.predict()`


## 1.2 .fit() — Method Fundamental

```python
estimator.fit(X, y=None)
```

* **Transformer** — contoh: `StandardScaler`, `OneHotEncoder`

  * Menggunakan `fit(X)`.
  * Hanya membutuhkan **fitur (`X`)**.
  * Tujuannya mempelajari statistik atau struktur dari data.
  * Tidak bertugas memprediksi target.

* **Predictor supervised** — contoh: `LogisticRegression`

  * Menggunakan `fit(X, y)`.
  * Membutuhkan **fitur (`X`) dan label (`y`)**.
  * Tujuannya mempelajari hubungan antara `X` dan `y`.

* **Model unsupervised** — contoh: `KMeans`

  * Menggunakan `fit(X)`.
  * Hanya membutuhkan **fitur (`X`)**.
  * Tidak membutuhkan `y` karena tidak terdapat label yang harus dipelajari.


* **`.fit()` selalu mengembalikan `self`**, yaitu objek estimator itu sendiri.
* `.fit()` **bukan** method untuk menghasilkan transformasi atau prediksi.
* Karena mengembalikan objek yang sama, method dapat langsung dirangkai:

```python
scaler = StandardScaler().fit(X_train)
```

Secara konsep:

```text
StandardScaler()
      ↓
    .fit(X)
      ↓
  self (scaler)
```

Jadi:

```python
scaler = StandardScaler().fit(X_train)
```

setara dengan:

```python
scaler = StandardScaler()
scaler.fit(X_train)
```

**Intinya:** `.fit()` → **belajar dari data → mengembalikan estimator yang sudah di-fit**.


```python
scaler = StandardScaler()
print(scaler.fit(X_train))
# Output: StandardScaler()   ← ini objeknya sendiri, bukan datanya

##  1.3 .transform() vs .fit_transform()

* `.transform(X)` → menerapkan aturan/statistik yang sudah dipelajari melalui `.fit()` sebelumnya.
* `.fit_transform(X)` → melakukan dua langkah sekaligus:

  1. Mempelajari aturan/statistik dari `X`.
  2. Langsung menerapkan hasil belajar tersebut pada `X`.

Secara konsep:

```python
transformer.fit(X)
X_transformed = transformer.transform(X)
```

setara dengan:

```python
X_transformed = transformer.fit_transform(X)
```

Intinya:

* `.fit()` → belajar
* `.transform()` → menerapkan hasil belajar
* `.fit_transform()` → belajar + menerapkan


* Untuk beberapa transformer, `.fit_transform()` dapat lebih efisien secara komputasi dibandingkan memanggil `.fit()` dan `.transform()` secara terpisah.
* Hal ini karena implementasi transformer tertentu dapat mengoptimalkan perhitungan antara proses belajar dan transformasi.
* Hasil akhirnya tetap sama secara konsep.
* Perbedaannya terletak pada **efisiensi proses di balik layar**, bukan pada hasil transformasinya.

Intinya:

```text
Training data → fit_transform()   (belajar DAN terapkan, sekaligus)
Testing data  → transform() SAJA   (terapkan HASIL BELAJAR dari training, JANGAN belajar lagi)
```


```pyhton
X_train_scaled = scaler.fit_transform(X_train)   # ✅ fit + transform di train
X_test_scaled = scaler.transform(X_test)          # ✅ transform SAJA di test
```

* Jangan menggunakan `scaler.fit_transform(X_test)`.
* `fit_transform()` akan:

  1. Menghitung mean dan standard deviation dari `X_test`.
  2. Menggunakan statistik tersebut untuk mentransformasi `X_test`.
* Ini menyebabkan scaler **belajar dari data test**.
* Seharusnya, statistik preprocessing hanya dipelajari dari `X_train`.
* Data test hanya boleh menggunakan hasil belajar tersebut:

```python
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Intinya:

```text
X_train → fit + transform
X_test  → transform saja
```

Jika `fit_transform(X_test)` digunakan, informasi dari test ikut masuk ke proses preprocessing → **data leakage**.
